# LightGBM 1차 검증 결과

저장된 실험 결과를 읽어 확인하는 노트북입니다. **모델 재학습·튜닝·최종 테스트를 실행하지 않습니다.**

- 실행 ID: `lightgbm_v001_initial_20261104_01`
- 대상: 20개국·8개 HS 제품의 분기 한국 수출금액 **USD·FOB**
- 검증 대상: 2022Q1~2025Q4, h=1~5 / 우선 비교 h=2~5
- 현재 수정치 기반 검증이며 과거 공개시점·빈티지를 재현한 검증이 아닙니다.
- A·C와 실제 공통 평가 코드·평가 행 대조는 별도 확인 사항입니다.

위에서부터 실행하면 전체 요약 → 국가·제품·거리별 지표 → 4분기 합계 → 행별 실제값·예측값 → 검증 기록 순으로 확인할 수 있습니다.

In [1]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display

RUN_ID = "lightgbm_v001_initial_20261104_01"
# 저장소 루트 또는 notebooks 폴더에서 실행할 수 있도록 상위 경로를 찾습니다.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "PROJECT_SPEC.md").is_file() and (p / "artifacts" / RUN_ID).is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("D:/five-stack 또는 notebooks 폴더에서 실행해 주세요.")
RUN = ROOT / "artifacts" / RUN_ID
pd.set_option("display.max_columns", 20)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")

metrics = pd.read_csv(RUN / "metrics.csv", dtype={"group_value": "string"})
common_metrics = pd.read_csv(RUN / "metrics_baseline_common_rows.csv", dtype={"group_value": "string"})
four_quarter_metrics = pd.read_csv(RUN / "four_quarter_metrics.csv")
predictions = pd.read_parquet(RUN / "validation_predictions.parquet")
metadata = json.loads((RUN / "run_metadata.json").read_text(encoding="utf-8"))
verification = json.loads((RUN / "verification.json").read_text(encoding="utf-8"))
LABELS = {"with_macro": "거시 포함", "without_macro": "거시 제외", "baseline": "Seasonal Naive"}
def metric_table(df):
    result = df.copy()
    if "variant" in result:
        result["variant"] = result["variant"].map(LABELS).fillna(result["variant"])
    return result.rename(columns={"variant":"구성", "n":"평가 건수", "wape":"WAPE (%)",
        "mae":"MAE (USD)", "rmse":"RMSE (USD)", "smape":"sMAPE (%)",
        "scope":"예측 거리 범위", "group_by":"집계 기준", "group_value":"집계 값", "window":"4분기 묶음"})
METRIC_COLUMNS = ["variant", "n", "wape", "mae", "rmse", "smape"]
print(f"프로젝트: {ROOT}")
print(f"결과 폴더: {RUN}")
print(f"전체 결과 행: {len(predictions):,} (두 LightGBM 구성과 기준선 포함)")


프로젝트: D:\five-stack
결과 폴더: D:\five-stack\artifacts\lightgbm_v001_initial_20261104_01
전체 결과 행: 34,365 (두 LightGBM 구성과 기준선 포함)


## 1. 운영 거리 h=2~5 — 동일 평가 행 비교

WAPE는 낮을수록 좋습니다. 아래는 두 LightGBM 구성의 **동일한 9,164개 행** 비교입니다. WAPE·sMAPE는 %, MAE·RMSE는 USD입니다. 이 결과로 운영 모델을 확정하거나 튜닝한 것은 아닙니다.

In [2]:
primary = metrics.loc[(metrics["scope"] == "h2_5") & (metrics["group_by"] == "overall")
                      & (metrics["model_id"] == "lightgbm")]
display(metric_table(primary[METRIC_COLUMNS]))
comparison = primary.set_index("variant")
gap = comparison.loc["with_macro", "wape"] - comparison.loc["without_macro", "wape"]
print(f"거시 포함 WAPE − 거시 제외 WAPE: {gap:.4f}%p")


,구성,평가 건수,WAPE (%),MAE (USD),RMSE (USD),sMAPE (%)
34,거시 포함,9164,27.4476,"2,958,972.9749","17,514,613.6003",69.7990
101,거시 제외,9164,26.8848,"2,898,299.1208","17,272,674.5249",68.8792


거시 포함 WAPE − 거시 제외 WAPE: 0.5628%p


## 2. Seasonal Naive와 공통 행 비교

기준선에 필요한 과거 관측값이 없는 행은 결측으로 남겼습니다. **표본 수가 다른 전체 결과를 바로 비교하지 않고**, 기준선까지 모두 예측 가능한 공통 행의 지표를 확인합니다. h=5도 origin 이후 실제값을 읽지 않는 방식입니다.

In [3]:
baseline_common = common_metrics.loc[(common_metrics["scope"] == "h2_5")
                                    & (common_metrics["group_by"] == "overall")]
display(metric_table(baseline_common[METRIC_COLUMNS]))


,구성,평가 건수,WAPE (%),MAE (USD),RMSE (USD),sMAPE (%)
34,거시 포함,9007,27.4291,"3,008,448.5844","17,666,588.5343",68.0509
101,거시 제외,9007,26.8660,"2,946,688.6962","17,422,550.2151",67.1097
168,Seasonal Naive,9007,29.6732,"3,254,582.2745","20,385,458.6437",62.0062


## 3. h=1~5 전체 요약과 예측 거리별 결과

첫 표는 전체 거리의 LightGBM 결과, 두 번째 표는 거리별 결과입니다. 동일 target이 서로 다른 origin·h의 예측 표본으로 반복되므로 행별 정답 합계를 국가의 실제 총수출액으로 해석하지 않습니다.

In [4]:
all_horizons = metrics.loc[(metrics["scope"] == "h1_5") & (metrics["group_by"] == "overall")
                           & (metrics["model_id"] == "lightgbm")]
display(metric_table(all_horizons[METRIC_COLUMNS]))
by_horizon = metrics.loc[(metrics["scope"] == "h1_5") & (metrics["group_by"] == "horizon_quarters")
                         & (metrics["model_id"] == "lightgbm")]
display(metric_table(by_horizon[["group_value", *METRIC_COLUMNS]].sort_values(["group_value", "variant"])))


,구성,평가 건수,WAPE (%),MAE (USD),RMSE (USD),sMAPE (%)
0,거시 포함,11455,25.6786,"2,768,267.6436","16,650,697.9355",69.0364
67,거시 제외,11455,25.1843,"2,714,976.1082","16,372,503.8957",68.3571


,집계 값,구성,평가 건수,WAPE (%),MAE (USD),RMSE (USD),sMAPE (%)
29,1,거시 포함,2291,18.6026,"2,005,446.3188","12,616,732.9537",65.9862
96,1,거시 제외,2291,18.3822,"1,981,684.0579","12,120,778.7745",66.2688
30,2,거시 포함,2291,21.9484,"2,366,135.3780","13,935,634.9388",67.7590
97,2,거시 제외,2291,21.8582,"2,356,407.0931","13,949,466.8943",67.6097
31,3,거시 포함,2291,25.3722,"2,735,238.8697","15,859,248.0052",68.9515
98,3,거시 제외,2291,25.0705,"2,702,714.8002","15,802,220.3025",67.9607
32,4,거시 포함,2291,29.3902,"3,168,398.6480","18,035,586.1461",70.6345
99,4,거시 제외,2291,28.7781,"3,102,411.8138","17,892,049.6777",69.1131
33,5,거시 포함,2291,33.0795,"3,566,119.0037","21,355,250.4599",71.8511
100,5,거시 제외,2291,31.8323,"3,431,662.7763","20,711,298.9980",70.8333


## 4. 국가별·제품별 성능

운영 거리 h=2~5만 비교합니다. 국가·제품별 WAPE의 단순 평균은 전체 WAPE와 다릅니다. 전체 WAPE는 전체 절대오차 합계와 실제값 합계에서 계산했습니다.

In [5]:
by_country = metrics.loc[(metrics["scope"] == "h2_5") & (metrics["group_by"] == "country_code")
                         & (metrics["model_id"] == "lightgbm")]
by_product = metrics.loc[(metrics["scope"] == "h2_5") & (metrics["group_by"] == "hs_code")
                         & (metrics["model_id"] == "lightgbm")]
print("국가별 WAPE (%)")
display(by_country.pivot(index="group_value", columns="variant", values="wape").rename(columns=LABELS))
print("HS 제품별 WAPE (%)")
display(by_product.pivot(index="group_value", columns="variant", values="wape").rename(columns=LABELS))


국가별 WAPE (%)


variant,거시 포함,거시 제외
group_value,,
AUS,22.0074,25.1746
BRA,39.5458,39.0640
CAN,26.1291,25.3835
CHN,29.7850,28.8532
DEU,27.4828,26.0602
ESP,25.6666,25.3654
FRA,30.6394,31.2267
GBR,27.6000,28.7046
GTM,304.3015,400.7999


HS 제품별 WAPE (%)


variant,거시 포함,거시 제외
group_value,,
330300,78.5457,72.6381
330410,37.9764,33.1667
330420,28.2401,27.4882
330491,58.1108,61.4114
330499,24.9628,24.6348
330510,51.4388,53.0956
330590,29.5971,29.0952
340130,38.2261,37.9203


## 5. 4분기 누적금액 오차

원본 파일: `four_quarter_metrics.csv`.

- `h1_4`: 같은 국가·제품·origin에서 h=1,2,3,4의 4개 분기 합계.
- `h2_5`: 같은 국가·제품·origin에서 h=2,3,4,5의 4개 분기 합계.
- 4개 정답·예측이 모두 있을 때만 계산합니다. **반드시 1~12월 달력연간 합계인 것은 아닙니다.**
- 기준선은 완전한 묶음의 개수가 다를 수 있으므로 아래 LightGBM 두 구성과 분리해 표시합니다. 기준선과 엄격한 공통 묶음 비교를 계산한 표는 아닙니다.

In [6]:
columns = ["variant", "window", "n", "wape", "mae", "rmse", "smape"]
display(metric_table(four_quarter_metrics.loc[four_quarter_metrics["model_id"] == "lightgbm", columns]))
print("기준선 참고 — 완전한 묶음 수가 다르므로 직접적인 우열 비교에 주의")
display(metric_table(four_quarter_metrics.loc[four_quarter_metrics["model_id"] == "seasonal_naive", columns]))


,구성,4분기 묶음,평가 건수,WAPE (%),MAE (USD),RMSE (USD),sMAPE (%)
0,거시 포함,h1_4,1773,21.0896,"9,511,412.1233","54,373,893.5241",58.3809
1,거시 포함,h2_5,1773,25.7932,"11,632,721.4388","67,214,672.2321",59.7395
2,거시 제외,h1_4,1773,21.2906,"9,602,059.3235","54,294,482.2266",57.9254
3,거시 제외,h2_5,1773,25.3366,"11,426,790.0085","66,567,082.1407",58.5665


기준선 참고 — 완전한 묶음 수가 다르므로 직접적인 우열 비교에 주의


,구성,4분기 묶음,평가 건수,WAPE (%),MAE (USD),RMSE (USD),sMAPE (%)
4,Seasonal Naive,h1_4,1759,24.2002,"11,000,820.0887","63,364,250.0008",47.7625
5,Seasonal Naive,h2_5,1756,29.0485,"13,227,164.1076","76,650,162.0464",53.4871


## 6. 행별 실제값·예측값 확인

아래 선택값을 바꾸고 셀을 다시 실행하면 원하는 결과를 볼 수 있습니다. `None`은 전체입니다. 예시는 국가 `USA`, HS `330499`처럼 문자열로 입력합니다. `variant`는 `with_macro`, `without_macro`, `baseline` 중 선택합니다. 표시 행 수만 제한하며 원본을 수정하지 않습니다.

In [7]:
country = None            # 예: "USA"
hs_code = None            # 예: "330499"
variant = "without_macro"
horizon = None            # 예: 2
display_rows = 30

selected = predictions.loc[predictions["variant"] == variant].copy()
if country is not None:
    selected = selected.loc[selected["country_code"] == country]
if hs_code is not None:
    selected = selected.loc[selected["hs_code"] == hs_code]
if horizon is not None:
    selected = selected.loc[selected["horizon_quarters"] == horizon]
selected["error_usd"] = selected["predicted_export_value_usd"] - selected["target_export_value_usd"]
columns = ["country_code", "hs_code", "origin_quarter", "target_quarter", "horizon_quarters",
           "variant", "target_export_value_usd", "predicted_export_value_usd", "error_usd", "prediction_status"]
print(f"선택 조건에 맞는 행: {len(selected):,}")
display(selected.sort_values(["country_code", "hs_code", "origin_quarter", "target_quarter"])[columns].head(display_rows))


선택 조건에 맞는 행: 11,455


,country_code,hs_code,origin_quarter,target_quarter,horizon_quarters,variant,target_export_value_usd,predicted_export_value_usd,error_usd,prediction_status
137,AUS,330300,2020Q4,2022Q1,5,without_macro,841.0000,"39,535.5023","38,694.5023",predicted
558,AUS,330300,2021Q1,2022Q1,4,without_macro,841.0000,"5,300.6736","4,459.6736",predicted
559,AUS,330300,2021Q1,2022Q2,5,without_macro,959.0000,"23,616.9084","22,657.9084",predicted
1267,AUS,330300,2021Q2,2022Q1,3,without_macro,841.0000,"59,493.2147","58,652.2147",predicted
1268,AUS,330300,2021Q2,2022Q2,4,without_macro,959.0000,"69,957.3950","68,998.3950",predicted
1269,AUS,330300,2021Q2,2022Q3,5,without_macro,"1,826.0000","89,656.9435","87,830.9435",predicted
2257,AUS,330300,2021Q3,2022Q1,2,without_macro,841.0000,0.0000,-841.0000,predicted
2258,AUS,330300,2021Q3,2022Q2,3,without_macro,959.0000,0.0000,-959.0000,predicted
2259,AUS,330300,2021Q3,2022Q3,4,without_macro,"1,826.0000",0.0000,"-1,826.0000",predicted
2260,AUS,330300,2021Q3,2022Q4,5,without_macro,"4,246.0000","21,645.2198","17,399.2198",predicted


## 7. 학습 범위·결측 제외·재현성 확인

검증 후보의 정답 결측 1,345개는 학습·정답 평가에서 제외했습니다. 각 origin의 과거 train 및 지난 validation 정답만 확장 학습한 내역이 `fit_audit.csv`에 있습니다. 공개일·빈티지는 미검증이므로 달력상 미래 정답 차단과 역사적 공개시점 재현은 구분합니다.

In [8]:
fit_audit = pd.read_csv(RUN / "fit_audit.csv")
display(fit_audit.loc[fit_audit["variant"] == "with_macro",
    ["origin_quarter", "fit_rows", "past_train_rows", "past_validation_rows", "max_fit_target", "score_rows"]])
checks = {
    "입력·설정 체크섬 보존": verification["protected_inputs_unchanged"],
    "두 구성 공통 유효 검증 행": verification["paired_validation_rows"],
    "h2~5 공통 평가 행": verification["primary_h2_5_rows"],
    "정답 결측 제외 행": verification["missing_target_rows"],
    "재로드 예측 일치 파이프라인": verification["pipelines_reloaded_exactly"],
    "테스트 통과 수": verification["unittest_count"],
    "최종 테스트 사용": verification["final_test_used"],
    "튜닝 수행": verification["tuning_performed"],
}
display(pd.DataFrame(checks.items(), columns=["확인 항목", "결과"]))
display(pd.DataFrame(metadata["libraries"].items(), columns=["라이브러리", "실행 시 버전"]))
print("이 표는 기존 실행에 저장된 검증 기록을 표시합니다. 노트북이 전체 테스트·학습을 다시 실행한 것은 아닙니다.")


,origin_quarter,fit_rows,past_train_rows,past_validation_rows,max_fit_target,score_rows
0,2020Q4,20565,20565,0,2020Q4,137
2,2021Q1,21225,21225,0,2021Q1,284
4,2021Q2,21910,21910,0,2021Q2,425
6,2021Q3,22605,22605,0,2021Q3,565
8,2021Q4,23290,23290,0,2021Q4,703
10,2022Q1,23975,23290,685,2022Q1,706
12,2022Q2,24710,23290,1420,2022Q2,704
14,2022Q3,25415,23290,2125,2022Q3,703
16,2022Q4,26115,23290,2825,2022Q4,705
18,2023Q1,26805,23290,3515,2023Q1,715


,확인 항목,결과
0,입력·설정 체크섬 보존,True
1,두 구성 공통 유효 검증 행,11455
2,h2~5 공통 평가 행,9164
3,정답 결측 제외 행,1345
4,재로드 예측 일치 파이프라인,40
5,테스트 통과 수,66
6,최종 테스트 사용,False
7,튜닝 수행,False


,라이브러리,실행 시 버전
0,lightgbm,4.5.0
1,scikit-learn,1.5.2
2,pandas,2.2.3
3,numpy,1.26.4
4,scipy,1.14.1
5,pyarrow,18.1.0
6,joblib,1.4.2
7,PyYAML,6.0.3


이 표는 기존 실행에 저장된 검증 기록을 표시합니다. 노트북이 전체 테스트·학습을 다시 실행한 것은 아닙니다.


## 원본 결과와 추가 확인

원본 CSV/Parquet·모델·변환기는 `artifacts/lightgbm_v001_initial_20261104_01/`에 있습니다. 상세 실행 방법은 해당 폴더의 `README_KO.md`와 `README.md`를 확인하세요. 이 노트북은 결과를 읽기만 하며 원본 지표·메타정보·체크섬을 수정하지 않습니다.